In [ ]:
# 读取json文件

import json

# 定义要读取的文件名和要读取的数据条数
file_path = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat/guichat_data.json'
num_to_read = 5  # 比如我们想读取前5条

try:
    # 使用 'with' 语句可以确保文件被正确关闭
    with open(file_path, 'r', encoding='utf-8') as f:
        # 使用 json.load() 从文件中加载整个JSON数据
        all_data = json.load(f)
        
        # 使用切片获取前 num_to_read 条数据
        first_few_items = all_data[:num_to_read]
        
        # 打印结果
        print(f"成功读取前 {len(first_few_items)} 条数据:")
        for item in first_few_items:
            print(item)

except FileNotFoundError:
    print(f"错误：找不到文件 '{file_path}'")
except json.JSONDecodeError:
    print(f"错误：文件 '{file_path}' 不是一个有效的JSON格式。")
except Exception as e:
    print(f"发生了一个错误: {e}")

错误：文件 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data_with_box.jsonl' 不是一个有效的JSON格式。


In [6]:
# 读取jsonl文件
import json

def read_first_n_lines_jsonl(file_path, n):
    """
    读取并解析一个 JSONL 文件的前 n 行。

    参数:
    file_path (str): JSONL 文件的路径。
    n (int): 需要读取的行数。

    返回:
    list: 一个包含已解析的JSON对象（字典）的列表。
    """
    data = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                # 如果已经读取了 n 行，则停止循环
                if i >= n:
                    break
                # 解析当前行并添加到列表中
                data.append(json.loads(line.strip()))
    except FileNotFoundError:
        print(f"错误：文件 '{file_path}' 未找到。")
    except json.JSONDecodeError as e:
        # 如果某一行不是有效的JSON，会捕获错误
        print(f"错误：解析第 {i+1} 行时发生JSON解码错误: {e}")
    except Exception as e:
        print(f"发生未知错误: {e}")
        
    return data




# 2. 设置文件路径和要读取的行数
file_path = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box_ratings.jsonl'
num_lines_to_read = 1

# 3. 调用函数读取文件
first_lines = read_first_n_lines_jsonl(file_path, num_lines_to_read)

# 4. 打印结果
if first_lines:
    print(f"成功读取文件 '{file_path}' 的前 {len(first_lines)} 行内容：")
    for item in first_lines:
        print(item)

成功读取文件 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box_ratings.jsonl' 的前 1 行内容：
{'id': 1, 'image_id': '050acf792a0c74036c3c0744c7ec50ee', 'task_type': 'single-turn-vqa', 'question': 'What are the incentives for playing the Aviator game at SOL casino?', 'ground_truth': "At SOL Casino, players who choose to play the Aviator game are offered a welcome bonus incentive. This bonus includes a 150% match on the amount they deposit plus up to 500 free spins (FS). The free spins package is credited to the player's account immediately after they activate the bonus. This offer is designed to attract new players to the casino and give them an enhanced gaming experience with the potential for more plays and increased chances of winning through the additional spins and bonus funds.", 'model_response': 'The incentives for playing the Aviator game at SOL casino include:\n\n1. **Welcome Bonus**: Players can receive a welcome bonus of up to 

In [ ]:
'''
Question: What are the incentives for playing the Aviator game at SOL casino?\nAnswer_with_box: At SOL Casino <box>[62,1231,187,1356]</box>, players who choose to play the Aviator game are offered a welcome bonus incentive <box>[378,1241,500,1269]</box>. This bonus includes a 150% match on the amount they deposit plus up to 500 free spins (FS) <box>[356,1290,521,1311]</box>. The free spins package is credited to the player's account immediately after they activate the bonus <box>[207,1330,669,1368]</box>. This offer is designed to attract new players to the casino and give them an enhanced gaming experience with the potential for more plays and increased chances of winning through the additional spins and bonus funds.
'''



In [ ]:
import json
import re

def process_json_to_jsonl(input_filename, output_filename):
    """
    读取一个JSON或JSONL文件，筛选和转换数据，并保存为新的JSONL文件。
    该版本修复了标点符号前多余空格的问题，并能正确移除Question中的<image>标签。
    """
    new_id = 1
    processed_count = 0

    try:
        with open(input_filename, 'r', encoding='utf-8') as f_in, \
             open(output_filename, 'w', encoding='utf-8') as f_out:
            
            # 智能判断输入文件是标准JSON还是JSONL
            first_char = f_in.read(1)
            f_in.seek(0)
            
            lines = []
            if first_char == '[':
                # 当作标准JSON处理
                try:
                    data = json.load(f_in)
                    lines = (json.dumps(item) for item in data) # 转换成可迭代的行
                except json.JSONDecodeError as e:
                    print(f"错误：文件看起来是标准JSON，但解析失败: {e}")
                    return
            else:
                # 当作JSONL处理
                lines = f_in

            for line in lines:
                if not line.strip():
                    continue
                
                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    print(f"警告：跳过无效的JSON行: {line.strip()}")
                    continue

                if item.get("task_type") == "single-turn-vqa":
                    question = None
                    answer = None

                    for text_part in item.get("text", []):
                        # --- 修改部分开始 ---
                        if text_part.get("from") == "human":
                            raw_question = text_part.get("value", "")
                            
                            # 步骤 1: 移除特定的模板句子 (忽略大小写)
                            template_pattern = r"Please grounding all object in this web page\.(?:\n|\s)*"
                            # re.IGNORECASE 使其能匹配 "please" 或 "Please"
                            temp_question = re.sub(template_pattern, "", raw_question, flags=re.IGNORECASE)
                            
                            # 步骤 2: 移除所有 <image>...</image> 标签
                            # '.*?' 是非贪婪匹配，确保只匹配到最近的'>'
                            image_tag_pattern = r"<image>.*?</image>"
                            temp_question = re.sub(image_tag_pattern, "", temp_question)
                            
                            # 步骤 3: 清理首尾和多余的空格
                            question = ' '.join(temp_question.split()).strip()
                        # --- 修改部分结束 ---

                        elif text_part.get("from") == "gpt":
                            raw_answer = text_part.get("value", "")
                            # 步骤 1: 移除 <box> 标签
                            answer_pattern = r"\s*<box>.*?</box>\s*"
                            cleaned_answer = re.sub(answer_pattern, " ", raw_answer).strip()
                            
                            # 步骤 2: 标准化空格（将多个空格合并为一个）
                            normalized_answer = ' '.join(cleaned_answer.split())
                            
                            # 步骤 3: 修复标点符号前的空格
                            punctuation_pattern = r'\s+([.,?!;:)\]}])'
                            final_answer = re.sub(punctuation_pattern, r'\1', normalized_answer)
                            answer = final_answer


                    if question is not None and answer is not None:
                        output_record = {
                            "id": new_id,
                            "image_id": item.get("image_id"),
                            "task_type": item.get("task_type"),
                            "Question": question,
                            "Answer": answer
                        }
                        f_out.write(json.dumps(output_record, ensure_ascii=False) + '\n')
                        new_id += 1
                        processed_count += 1
    
    except FileNotFoundError:
        print(f"错误：找不到输入文件 '{input_filename}'")
        return

    print(f"处理完成！共有 {processed_count} 条数据被写入到 '{output_filename}' 文件中。")


# --- 使用说明 ---
if __name__ == "__main__":
    # 1. 将你的JSON文件名修改为 'your_input_file.json'
    input_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat/guichat_data.json'
    
    # 2. 定义你希望生成的JSONL文件名
    output_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data.jsonl'
    
    # 3. 运行脚本
    process_json_to_jsonl(input_file, output_file)

# 这是原始的，把answer中的box去除，并且把question中的image信息去除

处理完成！共有 44728 条数据被写入到 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data.jsonl' 文件中。


In [9]:
import json
import re
from PIL import Image
import os # 引入os模块用于检查文件是否存在

def process_json_to_jsonl(input_filename, image_map_filename, output_filename):
    """
    读取JSON数据，根据图片尺寸转换坐标，并生成新的JSONL文件。
    - Answer: 清理后的纯文本回答。
    - Answer_with_box: 将<box>内的原始坐标替换为<box>[xmin,ymin,xmax,ymax]</box>格式的回答。
    """
    new_id = 1
    processed_count = 0
    
    # --- 1. 加载 image_id -> image_path 的映射文件 ---
    try:
        with open(image_map_filename, 'r', encoding='utf-8') as f:
            image_map = json.load(f)
    except FileNotFoundError:
        print(f"错误：找不到图片映射文件 '{image_map_filename}'")
        return
    except json.JSONDecodeError:
        print(f"错误：图片映射文件 '{image_map_filename}' 格式不正确。")
        return

    # 用于缓存图片尺寸，避免重复读取
    dim_cache = {}

    def convert_box_format(match, image_id):
        """
        一个 re.sub 的替换函数，用于转换单个 <box> 标签。
        """
        # --- 2. 获取图片尺寸 (带缓存) ---
        image_path = image_map.get(image_id)
        if not image_path:
            return "<box>[Image ID not found in map]</box>"
        
        if image_path in dim_cache:
            image_width, image_height = dim_cache[image_path]
        else:
            if not os.path.exists(image_path):
                 print(f"警告: 找不到图片文件 {image_path}，跳过此box的转换。")
                 return "<box>[Image file not found]</box>"
            try:
                with Image.open(image_path) as img:
                    image_width, image_height = img.size
                    dim_cache[image_path] = (image_width, image_height)
            except Exception as e:
                print(f"警告: 打开图片 {image_path} 失败: {e}，跳过此box的转换。")
                return "<box>[Image open failed]</box>"
        
        # 提取原始坐标文本 "x1 y1 x2 y2"
        coords_str = match.group(1)
        try:
            x1_orig, y1_orig, x2_orig, y2_orig = map(int, coords_str.split())
        except ValueError:
            return "<box>[Invalid coordinate format]</box>"

        # --- 3. 应用最终确定的坐标转换逻辑 ---
        x_final_top_left = (x1_orig / 1000.0) * image_width
        y_final_top_left = (y1_orig / 1000.0) * image_height
        abs_width = x2_orig - x1_orig
        abs_height = y2_orig - y1_orig
        x_final_bottom_right = x_final_top_left + abs_width
        y_final_bottom_right = y_final_top_left + abs_height

        # --- 4. 格式化为最终字符串，并用<box>标签包裹 (核心修改) ---
        return f"<box>[{int(x_final_top_left)},{int(y_final_top_left)},{int(x_final_bottom_right)},{int(y_final_bottom_right)}]</box>"

    try:
        with open(input_filename, 'r', encoding='utf-8') as f_in, \
             open(output_filename, 'w', encoding='utf-8') as f_out:
            
            first_char = f_in.read(1)
            f_in.seek(0)
            
            lines = []
            if first_char == '[':
                try:
                    data = json.load(f_in)
                    lines = (json.dumps(item) for item in data)
                except json.JSONDecodeError as e:
                    print(f"错误：文件看起来是标准JSON，但解析失败: {e}")
                    return
            else:
                lines = f_in

            for line in lines:
                if not line.strip():
                    continue
                
                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    print(f"警告：跳过无效的JSON行: {line.strip()}")
                    continue

                if item.get("task_type") == "single-turn-vqa":
                    question = None
                    answer = None
                    answer_with_box = None
                    image_id = item.get("image_id")

                    for text_part in item.get("text", []):
                        if text_part.get("from") == "human":
                            raw_question = text_part.get("value", "")
                            template_pattern = r"Please grounding all object in this web page\.(?:\n|\s)*"
                            temp_question = re.sub(template_pattern, "", raw_question, flags=re.IGNORECASE)
                            image_tag_pattern = r"<image>.*?</image>"
                            temp_question = re.sub(image_tag_pattern, "", temp_question)
                            question = ' '.join(temp_question.split()).strip()

                        elif text_part.get("from") == "gpt":
                            raw_answer = text_part.get("value", "")
                            
                            # --- 生成 Answer_with_box (新逻辑) ---
                            box_pattern_for_sub = r"<box>\s*([^<]*?)\s*</box>"
                            answer_with_box = re.sub(box_pattern_for_sub, lambda m: convert_box_format(m, image_id), raw_answer)

                            # --- 生成 Answer (旧逻辑) ---
                            answer_pattern_for_clean = r"\s*<box>.*?</box>\s*"
                            cleaned_answer = re.sub(answer_pattern_for_clean, " ", raw_answer).strip()
                            normalized_answer = ' '.join(cleaned_answer.split())
                            punctuation_pattern = r'\s+([.,?!;:)\]}])'
                            answer = re.sub(punctuation_pattern, r'\1', normalized_answer)

                    if question is not None and answer is not None:
                        output_record = {
                            "id": new_id,
                            "image_id": image_id,
                            "task_type": item.get("task_type"),
                            "Question": question,
                            "Answer": answer,
                            "Answer_with_box": answer_with_box # 添加新字段
                        }
                        f_out.write(json.dumps(output_record, ensure_ascii=False) + '\n')
                        new_id += 1
                        processed_count += 1
    
    except FileNotFoundError:
        print(f"错误：找不到输入文件 '{input_filename}'")
        return

    print(f"处理完成！共有 {processed_count} 条数据被写入到 '{output_filename}' 文件中。")


# --- 使用说明 ---
if __name__ == "__main__":
    # 1. 你的主要数据文件
    input_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat/guichat_data.json'
    
    # 2. 你的 image_id -> image_path 映射文件
    image_map_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/images/image_id2path.json'
    
    # 3. 你希望生成的JSONL文件名
    output_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data_with_box.jsonl'
    
    # 4. 运行脚本
    process_json_to_jsonl(input_file, image_map_file, output_file)

     # 这是含有Answer_with_box的jsonl文件，里面的box是转换后的正常的[xmin,ymin,xmax,ymax]格式


处理完成！共有 44728 条数据被写入到 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data_with_box.jsonl' 文件中。


In [1]:
import json
import re
import os
from PIL import Image, ImageDraw

def draw_bboxes_from_jsonl(data_file, image_map_file, output_dir, color='red', width=3):
    """
    读取JSONL文件，解析Answer_with_box字段中的坐标，
    并将所有边界框绘制到对应的原始图片上，然后根据id命名并保存到指定目录。

    参数:
    - data_file (str): 包含 box 坐标的 JSONL 文件 (singleturn_data_with_box.jsonl)。
    - image_map_file (str): image_id 到图片本地路径的映射 JSON 文件。
    - output_dir (str): 保存绘制后图片的文件夹路径。
    - color (str): 边界框的颜色。
    - width (int): 边界框的线宽。
    """
    # --- 1. 准备工作：创建输出目录和加载映射文件 ---
    print(f"准备工作开始...")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"已创建输出目录: {output_dir}")

    try:
        with open(image_map_file, 'r', encoding='utf-8') as f:
            image_map = json.load(f)
    except FileNotFoundError:
        print(f"错误: 找不到图片映射文件 '{image_map_file}'")
        return
    print("图片路径映射加载成功。")

    # --- 2. 逐行处理JSONL文件 ---
    print(f"\n开始处理文件: {data_file}")
    processed_count = 0
    total_boxes_drawn = 0
    
    try:
        with open(data_file, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f, 1):
                try:
                    item = json.loads(line)
                    image_id = item.get('image_id')
                    # 修改：读取 Answer_with_box 字段
                    answer_with_box = item.get('Answer_with_box')
                    # 新增：获取 id (在您的生成脚本中字段名为 'id')
                    item_id = item.get('id')

                    if not image_id or not answer_with_box or item_id is None:
                        print(f"警告: 第 {i} 行缺少 'id', 'image_id' 或 'Answer_with_box'，已跳过。")
                        continue

                    # --- 3. 解析所有 box 坐标 ---
                    box_pattern = re.compile(r"<box>\[(\d+,\d+,\d+,\d+)]</box>")
                    coord_strings = box_pattern.findall(answer_with_box)

                    if not coord_strings:
                        continue 

                    bboxes_list = []
                    for cs in coord_strings:
                        bboxes_list.append(list(map(int, cs.split(','))))

                    # --- 4. 绘制并保存图片 ---
                    original_image_path = image_map.get(image_id)
                    if not original_image_path or not os.path.exists(original_image_path):
                        print(f"警告: 找不到 image_id '{image_id}' 对应的图片文件，已跳过。")
                        continue
                    
                    try:
                        with Image.open(original_image_path).convert("RGB") as image:
                            draw = ImageDraw.Draw(image)
                            
                            for bbox in bboxes_list:
                                draw.rectangle([(bbox[0], bbox[1]), (bbox[2], bbox[3])], outline=color, width=width)
                            
                            # 修改：使用 item_id 来命名输出文件
                            output_filename = f"{item_id}_box.png"
                            output_path = os.path.join(output_dir, output_filename)
                            
                            image.save(output_path)
                            processed_count += 1
                            total_boxes_drawn += len(bboxes_list)
                            if processed_count % 100 == 0:
                                print(f"已处理 {processed_count} 张图片...")

                    except Exception as e:
                        print(f"错误: 处理图片 '{original_image_path}' 时发生错误: {e}")

                except json.JSONDecodeError:
                    print(f"警告: 第 {i} 行不是有效的JSON格式，已跳过。")

    except FileNotFoundError:
        print(f"错误: 找不到输入数据文件 '{data_file}'")
        return

    print("\n处理完成！")
    print(f"共成功生成了 {processed_count} 张带有边界框的图片。")
    print(f"总计绘制了 {total_boxes_drawn} 个边界框。")
    print(f"结果已全部保存到目录: '{output_dir}'")


# --- 使用说明 ---
if __name__ == "__main__":
    # 1. 你的数据文件
    # 修改：指定新的输入文件
    input_data_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data_with_box.jsonl'
    
    # 2. 你的 image_id -> image_path 映射文件
    image_map_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/images/image_id2path.json' 
    
    # 3. 你希望保存结果的文件夹
    output_directory = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/images_with_boxes_40k' 
    
    # 4. 运行主函数
    draw_bboxes_from_jsonl(input_data_file, image_map_file, output_directory)



准备工作开始...
图片路径映射加载成功。

开始处理文件: /mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data_with_box.jsonl
已处理 100 张图片...
已处理 200 张图片...
已处理 300 张图片...
已处理 400 张图片...
已处理 500 张图片...
错误: 处理图片 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/images/chunk_9/849584cedb3b18bf354f785d5fa98cb8.png' 时发生错误: y1 must be greater than or equal to y0
已处理 600 张图片...
已处理 700 张图片...
已处理 800 张图片...
已处理 900 张图片...
已处理 1000 张图片...
已处理 1100 张图片...
已处理 1200 张图片...
错误: 处理图片 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/images/chunk_3/31dfd17b40e0d328b66e0f0c7f8a7118.png' 时发生错误: x1 must be greater than or equal to x0
已处理 1300 张图片...
已处理 1400 张图片...
已处理 1500 张图片...
错误: 处理图片 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/images/chunk_17/f2fd8d9f23d0494a26fefff1e5ae033d.png' 时发生错误: x1 must be greater than or equal to x0
已处理 1600 张图片...
已处理 1700 张图片...
已处理 180

In [2]:
import json

def filter_and_reindex_jsonl(input_file_path, output_file_path):
    """
    读取一个 JSONL 文件，筛选出 'llm_score' 为 0 的条目，
    并用从 1 开始的新 ID 将它们保存到新的 JSONL 文件中。

    Args:
        input_file_path (str): 输入的 JSONL 文件的路径。
        output_file_path (str): 输出的 JSONL 文件的路径。
    """
    # 用于存储筛选后的数据
    filtered_data = []
    # 新的 ID 从 1 开始
    new_id = 1

    try:
        with open(input_file_path, 'r', encoding='utf-8') as infile:
            # 跳过第一行
            next(infile, None)
            
            # 逐行读取和处理
            for line_num, line in enumerate(infile, 2): # 从第 2 行开始计数
                # 去除行尾的换行符
                line = line.strip()
                if not line:
                    continue

                try:
                    # 解析 JSON 数据
                    data = json.loads(line)
                    
                    # 检查 llm_score 是否为 0
                    if data.get('llm_score') == 0:
                        # 更新 id
                        data['id'] = new_id
                        # 将修改后的数据添加到列表中
                        filtered_data.append(data)
                        # 递增新 id
                        new_id += 1
                
                except json.JSONDecodeError:
                    print(f"警告: 第 {line_num} 行不是有效的 JSON 格式，已跳过。内容: '{line}'")
                except KeyError:
                    print(f"警告: 第 {line_num} 行缺少 'llm_score' 键，已跳过。")

        # 将筛选后的数据写入新的 JSONL 文件
        with open(output_file_path, 'w', encoding='utf-8') as outfile:
            for item in filtered_data:
                # 将字典转换为 JSON 字符串并写入文件，确保每条记录占一行
                outfile.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        print(f"处理完成！")
        print(f"总共筛选出 {len(filtered_data)} 条 'llm_score' 为 0 的数据。")
        print(f"结果已保存至: {output_file_path}")

    except FileNotFoundError:
        print(f"错误: 输入文件未找到 -> {input_file_path}")
    except Exception as e:
        print(f"处理过程中发生未知错误: {e}")

# --- 使用示例 ---
# 请将 'your_input_file.jsonl' 替换为您的原始文件名
# 请将 'output_file.jsonl' 替换为您想要保存的新文件名
input_filename = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm.jsonl'
output_filename = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered.jsonl'

# 调用函数
filter_and_reindex_jsonl(input_filename, output_filename)
# 根据llm_score过滤

处理完成！
总共筛选出 7615 条 'llm_score' 为 0 的数据。
结果已保存至: /mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered.jsonl


In [ ]:
import json

def merge_data_with_box(source_file, target_file, output_file):
    """
    将带有box标注的答案合并到目标数据文件中。

    它通过 (image_id, question) 作为唯一键，从源文件中查找
    'Answer_with_box'，并将其作为 'ground_truth_with_box' 字段
    添加到目标文件的每一行中。

    参数:
    - source_file (str): 包含 'Answer_with_box' 的源文件 (e.g., singleturn_data_with_box.jsonl)。
    - target_file (str): 需要被添加新字段的目标文件 (e.g., 72b_responses_llm_filtered.jsonl)。
    - output_file (str): 最终生成的合并后的文件。
    """
    # 步骤 1: 从源文件构建查找字典
    print(f"正在从 '{source_file}' 构建查找数据...")
    lookup_data = {}
    try:
        with open(source_file, 'r', encoding='utf-8') as f_source:
            for line in f_source:
                item = json.loads(line)
                # 使用 (image_id, Question) 元组作为唯一的键，这比字符串拼接更可靠
                key = (item.get('image_id'), item.get('Question'))
                if key[0] and key[1]:
                    lookup_data[key] = item.get('Answer_with_box')
    except FileNotFoundError:
        print(f"错误: 源文件 '{source_file}' 未找到。")
        return

    print(f"查找数据构建完成，共 {len(lookup_data)} 条记录。")

    # 步骤 2: 处理目标文件并合并数据
    print(f"正在处理 '{target_file}' 并合并数据...")
    processed_count = 0
    match_count = 0
    try:
        with open(target_file, 'r', encoding='utf-8') as f_target, \
             open(output_file, 'w', encoding='utf-8') as f_out:
            for line in f_target:
                target_item = json.loads(line)

                # 从目标文件中创建用于查找的键
                # 注意目标文件中的字段名是 'question'
                key_to_find = (target_item.get('image_id'), target_item.get('question'))

                # 从字典中查找对应的 'Answer_with_box'
                ground_truth_with_box = lookup_data.get(key_to_find)

                if ground_truth_with_box is not None:
                    match_count += 1
                else:
                    # 如果找不到匹配项，打印警告，并使用原始的 ground_truth 作为默认值
                    print(f"警告: 未找到匹配项 -> image_id: {key_to_find[0]}, question: '{key_to_find[1][:50]}...'")
                    ground_truth_with_box = target_item.get('ground_truth') 

                # 在原始数据基础上，添加新的字段
                target_item['ground_truth_with_box'] = ground_truth_with_box

                # 将更新后的记录写入新文件
                f_out.write(json.dumps(target_item, ensure_ascii=False) + '\n')
                processed_count += 1
    except FileNotFoundError:
        print(f"错误: 目标文件 '{target_file}' 未找到。")
        return

    print("\n处理完成！")
    print(f"共处理了 {processed_count} 条记录。")
    print(f"成功匹配并添加 'ground_truth_with_box' 的记录有 {match_count} 条。")
    print(f"结果已保存到 '{output_file}'。")

# --- 使用说明 ---
if __name__ == "__main__":
    # 1. 源文件：包含我们需要的 Answer_with_box 字段
    source_data_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data_with_box.jsonl'

    # 2. 目标文件：我们需要向这个文件中添加新字段
    target_data_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered.jsonl'

    # 3. 输出文件：最终生成的结果文件
    output_data_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box.jsonl'
    
    # 4. 运行主函数
    merge_data_with_box(source_data_file, target_data_file, output_data_file)

# 创建含有Answer_with_box的llm_score为0的数据

正在从 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/singleturn_data_with_box.jsonl' 构建查找数据...
查找数据构建完成，共 44727 条记录。
正在处理 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_right.jsonl' 并合并数据...

处理完成！
共处理了 37113 条记录。
成功匹配并添加 'ground_truth_with_box' 的记录有 37113 条。
结果已保存到 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box_right.jsonl'。


In [4]:
import json
import re
import os
from PIL import Image, ImageDraw

def draw_bboxes_from_jsonl(data_file, image_map_file, output_dir, color='red', width=3):
    """
    读取JSONL文件，解析ground_truth_with_box字段中的坐标，
    并将所有边界框绘制到对应的原始图片上，然后根据id命名并保存到指定目录。

    参数:
    - data_file (str): 包含 box 坐标的 JSONL 文件 (e.g., 72b_responses_llm_filtered_box.jsonl)。
    - image_map_file (str): image_id 到图片本地路径的映射 JSON 文件。
    - output_dir (str): 保存绘制后图片的文件夹路径。
    - color (str): 边界框的颜色。
    - width (int): 边界框的线宽。
    """
    # --- 1. 准备工作：创建输出目录和加载映射文件 ---
    print(f"准备工作开始...")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"已创建输出目录: {output_dir}")

    try:
        with open(image_map_file, 'r', encoding='utf-8') as f:
            image_map = json.load(f)
    except FileNotFoundError:
        print(f"错误: 找不到图片映射文件 '{image_map_file}'")
        return
    print("图片路径映射加载成功。")

    # --- 2. 逐行处理JSONL文件 ---
    print(f"\n开始处理文件: {data_file}")
    processed_count = 0
    total_boxes_drawn = 0
    
    try:
        with open(data_file, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f, 1):
                try:
                    item = json.loads(line)
                    image_id = item.get('image_id')
                    gt_with_box = item.get('ground_truth_with_box')
                    # 新增：获取 id
                    item_id = item.get('id')

                    # 修改：在检查条件中加入 item_id
                    if not image_id or not gt_with_box or item_id is None:
                        print(f"警告: 第 {i} 行缺少 'id', 'image_id' 或 'ground_truth_with_box'，已跳过。")
                        continue

                    # --- 3. 解析所有 box 坐标 ---
                    box_pattern = re.compile(r"<box>\[(\d+,\d+,\d+,\d+)]</box>")
                    coord_strings = box_pattern.findall(gt_with_box)

                    if not coord_strings:
                        continue 

                    bboxes_list = []
                    for cs in coord_strings:
                        bboxes_list.append(list(map(int, cs.split(','))))

                    # --- 4. 绘制并保存图片 ---
                    original_image_path = image_map.get(image_id)
                    if not original_image_path or not os.path.exists(original_image_path):
                        print(f"警告: 找不到 image_id '{image_id}' 对应的图片文件，已跳过。")
                        continue
                    
                    try:
                        with Image.open(original_image_path).convert("RGB") as image:
                            draw = ImageDraw.Draw(image)
                            
                            for bbox in bboxes_list:
                                draw.rectangle([(bbox[0], bbox[1]), (bbox[2], bbox[3])], outline=color, width=width)
                            
                            # 修改：使用 item_id 来命名输出文件
                            output_filename = f"{item_id}_box.png"
                            output_path = os.path.join(output_dir, output_filename)
                            
                            image.save(output_path)
                            processed_count += 1
                            total_boxes_drawn += len(bboxes_list)
                            if processed_count % 100 == 0:
                                print(f"已处理 {processed_count} 张图片...")

                    except Exception as e:
                        print(f"错误: 处理图片 '{original_image_path}' 时发生错误: {e}")

                except json.JSONDecodeError:
                    print(f"警告: 第 {i} 行不是有效的JSON格式，已跳过。")

    except FileNotFoundError:
        print(f"错误: 找不到输入数据文件 '{data_file}'")
        return

    print("\n处理完成！")
    print(f"共成功生成了 {processed_count} 张带有边界框的图片。")
    print(f"总计绘制了 {total_boxes_drawn} 个边界框。")
    print(f"结果已全部保存到目录: '{output_dir}'")


# --- 使用说明 ---
if __name__ == "__main__":
    # 1. 你的数据文件
    input_data_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box.jsonl'
    
    # 2. 你的 image_id -> image_path 映射文件
    image_map_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/images/image_id2path.json'
    
    # 3. 你希望保存结果的文件夹
    output_directory = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/images_with_boxes'
    
    # 4. 运行主函数
    draw_bboxes_from_jsonl(input_data_file, image_map_file, output_directory)



准备工作开始...
已创建输出目录: /mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/images_with_boxes
图片路径映射加载成功。

开始处理文件: /mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box.jsonl
已处理 100 张图片...
已处理 200 张图片...
已处理 300 张图片...
已处理 400 张图片...
已处理 500 张图片...
已处理 600 张图片...
已处理 700 张图片...
已处理 800 张图片...
已处理 900 张图片...
已处理 1000 张图片...
已处理 1100 张图片...
已处理 1200 张图片...
已处理 1300 张图片...
已处理 1400 张图片...
已处理 1500 张图片...
已处理 1600 张图片...
已处理 1700 张图片...
已处理 1800 张图片...
已处理 1900 张图片...
已处理 2000 张图片...
已处理 2100 张图片...
错误: 处理图片 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/images/chunk_6/5a4a7ddb3d2fd94911fab9cc4f0ed053.png' 时发生错误: x1 must be greater than or equal to x0
已处理 2200 张图片...
已处理 2300 张图片...
已处理 2400 张图片...
已处理 2500 张图片...
已处理 2600 张图片...
已处理 2700 张图片...
已处理 2800 张图片...
已处理 2900 张图片...
已处理 3000 张图片...
已处理 3100 张图片...
已处理 3200 张图片...
已处理 3300 张图片...
已处理 3400 张图片...
已处理 3500 张图片...
已处

In [7]:
import json
import os

def filter_data_by_score(ratings_file, data_file, output_file):
    """
    根据评分文件筛选主数据文件。

    参数:
    - ratings_file (str): 包含评分的JSONL文件 (e.g., rating.jsonl)。
    - data_file (str): 原始数据来源的JSONL文件 (e.g., 72b_responses_llm_filtered_box.jsonl)。
    - output_file (str): 保存筛选后数据的目标JSONL文件。
    """
    # --- 1. 从评分文件中提取所有 score 为 1 的 ID ---
    print(f"正在从 '{ratings_file}' 读取评分...")
    ids_with_score_one = set()
    try:
        with open(ratings_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    # 检查 score 是否为 1
                    if item.get('score') == 1:
                        filename = item.get('filename', '')
                        # 从 "10_box.png" 中提取出 ID "10"
                        if filename and '_' in filename:
                            item_id_str = filename.split('_')[0]
                            ids_with_score_one.add(int(item_id_str))

                except (json.JSONDecodeError, ValueError, IndexError) as e:
                    print(f"警告: 解析评分文件中的行时出错: {line.strip()} - {e}")

    except FileNotFoundError:
        print(f"错误: 找不到评分文件 '{ratings_file}'")
        return
    
    if not ids_with_score_one:
        print("警告: 在评分文件中没有找到任何 score 为 1 的数据。")
        return
        
    print(f"找到 {len(ids_with_score_one)} 个 score 为 1 的 ID。")

    # --- 2. 遍历主数据文件，根据 ID 列表进行筛选 ---
    print(f"\n正在从 '{data_file}' 筛选数据...")
    written_count = 0
    try:
        with open(data_file, 'r', encoding='utf-8') as f_in, \
             open(output_file, 'w', encoding='utf-8') as f_out:
            
            for line in f_in:
                try:
                    item = json.loads(line)
                    # 检查当前数据的 id 是否在我们收集的集合中
                    if item.get('id') in ids_with_score_one:
                        # 如果是，则将该行完整写入新文件
                        f_out.write(json.dumps(item, ensure_ascii=False) + '\n')
                        written_count += 1
                
                except json.JSONDecodeError as e:
                     print(f"警告: 解析主数据文件中的行时出错: {line.strip()} - {e}")


    except FileNotFoundError:
        print(f"错误: 找不到主数据文件 '{data_file}'")
        return

    print("\n处理完成！")
    print(f"共筛选出 {written_count} 条数据，并已保存到 '{output_file}' 文件中。")


# --- 使用说明 ---
if __name__ == "__main__":
    # 1. 你的评分文件
    ratings_filename = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/ratings.jsonl'
    
    # 2. 你的主数据文件
    main_data_filename = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box.jsonl'
    
    # 3. 你希望生成的新文件名
    output_filename = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box_ratings.jsonl'
    
    # 4. 运行主函数
    filter_data_by_score(ratings_filename, main_data_filename, output_filename)


正在从 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/ratings.jsonl' 读取评分...
找到 1870 个 score 为 1 的 ID。

正在从 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box.jsonl' 筛选数据...

处理完成！
共筛选出 1870 条数据，并已保存到 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box_ratings.jsonl' 文件中。


In [11]:
import json
import pandas as pd
from PIL import Image
import io

def convert_jsonl_to_parquet(jsonl_file_path, image_map_json_path, output_parquet_path):
    """
    将 JSONL 文件和图像数据转换为 Parquet 文件。

    Args:
        jsonl_file_path (str): 输入的 JSONL 文件的路径。
        image_map_json_path (str): 包含 image_id 到图像路径映射的 JSON 文件。
        output_parquet_path (str): 输出的 Parquet 文件的路径。
    """
    try:
        # 1. 读取 image_id 到图像路径的映射
        with open(image_map_json_path, 'r', encoding='utf-8') as f:
            image_id_to_path = json.load(f)
        print(f"成功加载 {len(image_id_to_path)} 条图像路径映射。")

        # 用于存储处理后数据的列表
        data_for_df = []

        # limit_num = 20
        start_line = 1230

        # 2. 逐行读取 JSONL 文件
        with open(jsonl_file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                if line_num < start_line:
                    continue

                line = line.strip()
                if not line:
                    continue
                
                try:
                    record = json.loads(line)
                    
                    image_id = record.get('image_id')
                    question = record.get('question')
                    answer = record.get('ground_truth')
                    answer_with_box = record.get('ground_truth_with_box')

                    if not all([image_id, question, answer]):
                        print(f"警告: 第 {line_num} 行缺少必要的字段，已跳过。")
                        continue

                    # 3. 查找图像路径并读取图像
                    image_path = image_id_to_path.get(image_id)
                    if not image_path:
                        print(f"警告: 在图像映射中未找到 image_id '{image_id}'，已跳过第 {line_num} 行。")
                        continue

                    try:
                        # 读取图像文件为二进制数据
                        with open(image_path, 'rb') as img_file:
                            image_bytes = img_file.read()

                        question_answer_with_box = "Question: " + question + "\nAnswer_with_box: " + answer_with_box

                        # 将所有数据添加到列表中
                        data_for_df.append({
                            'image': image_bytes,
                            'question': question,
                            'answer': answer,
                            'question_answer_with_box': question_answer_with_box,
                        })
                    except FileNotFoundError:
                        print(f"警告: 图像文件未找到: {image_path}，已跳过第 {line_num} 行。")
                    except Exception as img_e:
                        print(f"警告: 读取图像 {image_path} 时出错: {img_e}，已跳过第 {line_num} 行。")

                except json.JSONDecodeError:
                    print(f"警告: 第 {line_num} 行不是有效的 JSON 格式，已跳过。")

        if not data_for_df:
            print("没有可处理的数据，无法生成 Parquet 文件。")
            return

        # 4. 创建 Pandas DataFrame
        df = pd.DataFrame(data_for_df)
        print(f"\n成功创建 DataFrame，包含 {len(df)} 条记录。")

        # 5. 保存为 Parquet 文件
        df.to_parquet(output_parquet_path, engine='pyarrow')
        print(f"处理完成！数据已成功保存至: {output_parquet_path}")

    except FileNotFoundError as e:
        print(f"错误: 文件未找到 -> {e.filename}")
    except Exception as e:
        print(f"处理过程中发生未知错误: {e}")


# --- 使用示例 ---
# 请将 'filtered_output.jsonl' 替换为您在上一步生成的 JSONL 文件名
jsonl_filename = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box_ratings.jsonl'

# 请将 'image_paths.json' 替换为包含图像 ID 到路径映射的 JSON 文件名
image_map_filename = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/datasets/GUIChat-processed/images/image_id2path.json' 

# 这是您想要生成的 Parquet 文件名
output_filename = '/mnt/petrelfs/sunhaoyu/visual-code/datasets/GUIChat/sft_v1/validation.parquet'

# 调用函数
# 为了Validation，我只构造了20条数据，称之为Validation集
# 设置了一个limit_num，不想使用可以删除
convert_jsonl_to_parquet(jsonl_filename, image_map_filename, output_filename)

成功加载 17979 条图像路径映射。

成功创建 DataFrame，包含 641 条记录。
处理完成！数据已成功保存至: /mnt/petrelfs/sunhaoyu/visual-code/datasets/GUIChat/sft_v1/validation.parquet
